# NeurIPS 2026 Smart Buildings — Version 12: Efficient MPC (compact V6)

**Experiment:** Reduce the Version 6 action-conditioned planning workload without changing its core online-learning hypothesis. **Version 6 timed out on Kaggle after >9 hours; Version 12 is unscored.** Lower competition scores are better.

## Control pipeline

1. Parse metadata-driven AHU/VAV sensors and denormalize room temperatures.
2. Update each zone's bounded four-coefficient RLS thermal model using **the preceding issued action**.
3. Reject unsafe plant modes, then choose among **off, cooling, heating and mixed**.
4. For each mode, score at most **three actuator-reachable damper/reheat profiles per zone**, projecting the learned five-minute thermal response to **20 and 50 minutes**.
5. Use scalar scoring and a nonnegative-cost early cutoff to avoid redundant work.
6. Apply AHU/VAV commands, coordinate the shared boiler/pump, and save features for the next five-minute observation.

## Search size and rationale

| Item | Version 6 | Version 12 |
| --- | ---: | ---: |
| AHU modes | 7 | 4 |
| Maximum VAV profiles evaluated per zone across all modes | 49 | 10 |
| Horizon checkpoints per profile | 3 (20/40/60 min) | 2 (20/50 min) |
| Worst-case comfort evaluations per zone before pruning | 147 | 20 |
| Action-conditioned RLS identification | Yes | Preserved |
| Comfort and energy surrogate | Yes | Preserved |
| Severe hot/cold and simultaneous heat/cool guardrails | Yes | Preserved |
| Idle/plant command safety and hot-water coordination | Yes | Preserved |

The forecast-evaluation count is **approximately 7.35× smaller**, before early pruning. This is a computational search reduction, **not** a measured runtime speedup or Kaggle-score improvement.

## Component map

| Component | Responsibility |
| --- | --- |
| `ZoneResponseModel` | Online RLS update, bounded heating/cooling coefficients, previous-action feature memory |
| `TransferPolicy.__init__`, `_read_zones` | Parse dynamic action/sensor indices, normalization, and invalid/missing zone readings |
| `OPERATING_POINTS`, `_candidate_allowed` | Four operating points; forbid unsafe off or one-sided modes |
| `_profiles`, `_best_vav` | Small command grid and scalar horizon scoring with actuator availability |
| `_select_operating_point` | Minimize averaged zone cost plus plant cost, with safe early cutoff |
| `_apply_ahu_action`, `_apply_vav_actions` | Translate physical choices into normalized commands and retain learning features |
| `_apply_shared_hot_water` | Switch on the shared water loop only when a valid reheat request exists |
| `get_action`, `reset` | Run one decision and isolate successive episodes |

## Explicit limitations and validation

- The model assumes an approximately constant five-minute temperature response over 20–50 minutes; **it is not a full multi-step physics simulator**. Energy and comfort are heuristic surrogates, not the exact competition objective.
- Reduced action resolution could lose useful intermediate SAT, pressure or reheat settings. The fixed modes are a deliberate speed/quality trade-off, not a claim of dominance over V6 or V7.
- The VAV grid respects unavailable damper/valve commands; outdoor-air observations are denormalized using the supplied sensor statistics. The pending RLS features use the **shared** boiler setpoint after AHU aggregation.
- Scalar profile scores match the V6 action-feature equation for retained candidates. RLS covariance is bounded for long episodes to prevent unexcited directions from growing without limit.
- `test_version12.py` tests synthetic contract, safety, reset, scalar-model equivalence and latency. **Official 30-day simulator scoring is still required**; no public or hidden score is claimed.
- The last notebook cell uses the organizer's `SmartBuildingsInferenceServer(TransferPolicy).run()` entrypoint.


In [ ]:
"""Version 12: compact action-conditioned MPC derived from Version 6.

Preserves per-zone online RLS identification, model-based comfort/energy
scoring, severe-comfort safeguards, and the Kaggle hybrid-action contract.
Reduces mode/profile combinations and forecast checkpoints, uses scalar
per-profile evaluation, and stops a candidate early when it cannot win.
This is approximate receding-horizon planning, not a simulator rollout.
"""
import re
from typing import NamedTuple

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class OperatingPoint(NamedTuple):
    """One physical AHU/plant operating point in Kelvin and pascals."""

    mode: str
    running: bool
    supply_k: float
    fan_pa: float
    water_k: float
    water_on: bool


class ZoneResponseModel:
    """Fit one zone's five-minute Kelvin change using bounded online RLS.

    Regression: ΔT = b0 * ((Tout-T)/20) + b1 * cooling + b2 * reheat + b3.
    The features encode *issued actions*, so the model can compare the
    consequences of candidate commands, unlike a passive trend estimator.
    Coefficient signs are constrained by physical direction; the priors are
    used from step one and the covariance limits the size of updates.
    """

    MAX_COVARIANCE_DIAGONAL = 8.0
    INITIAL_COVARIANCE = (0.45, 0.55, 0.55, 0.12)

    def __init__(self):
        self.weights = np.array([0.035, -0.16, 0.20, 0.0], dtype=np.float64)
        self.covariance = np.diag(self.INITIAL_COVARIANCE)
        self.last_temperature = None
        self.last_features = None
        self.samples = 0

    @staticmethod
    def features(temp, outdoor, run, supply, damper, valve, water_on, water_temp):
        """Produce action-dependent ambient/cooling/reheat/bias features.

        Temperatures use Kelvin; damper/valve are fractions, not commands.
        """
        outdoor = outdoor if np.isfinite(outdoor) else temp
        ambient = np.clip((outdoor - temp) / 20.0, -3.0, 3.0)
        # AHU cooling coil cannot heat. Natural outdoor mixing can cool air
        # further than the supply setpoint even with the chiller inactive.
        mixed_air = 0.70 * temp + 0.30 * outdoor
        delivered = min(supply, mixed_air)
        cooling = run * damper * np.clip((temp - delivered) / 12.0, 0.0, 2.0)
        reheat = (run * damper * valve * water_on *
                  np.clip((water_temp - temp) / 55.0, 0.0, 2.0))
        return np.array([ambient, cooling, reheat, 1.0], dtype=np.float64)

    def observe(self, current_temperature):
        """Update RLS coefficients from the previous issued action's outcome.

        Forgetting factor 0.995; clip residual and physically directional
        coefficients; discard implausible >1.5K five-minute jumps.
        """
        if self.last_temperature is None or self.last_features is None:
            return
        delta = current_temperature - self.last_temperature
        if not np.isfinite(delta) or abs(delta) > 1.5:
            return  # sensor discontinuity, not ordinary thermal response
        x = self.last_features
        px = self.covariance @ x
        denominator = 0.995 + x @ px
        if not np.isfinite(denominator) or denominator <= 1e-10:
            # Recover from ill-conditioned identification rather than letting
            # one numerical failure contaminate every later action.
            self.covariance = np.diag(self.INITIAL_COVARIANCE)
            return

        gain = px / denominator
        predicted = float(self.weights @ x)
        self.weights += gain * np.clip(delta - predicted, -0.8, 0.8)
        self.weights = np.clip(
            self.weights,
            [0.002, -0.65, 0.004, -0.12],
            [0.20, -0.004, 0.70, 0.12],
        )

        self.covariance = (self.covariance - np.outer(gain, px)) / 0.995
        self.covariance = 0.5 * (self.covariance + self.covariance.T)

        # RLS forgetting otherwise inflates unexcited covariance directions
        # by 1/0.995 at every step; this episode lasts 8,640 steps.
        diagonal = np.diag(self.covariance)
        if (not np.all(np.isfinite(self.covariance)) or
                np.any(diagonal <= 0.0)):
            self.covariance = np.diag(self.INITIAL_COVARIANCE)
        else:
            largest = float(np.max(diagonal))
            if largest > self.MAX_COVARIANCE_DIAGONAL:
                self.covariance *= self.MAX_COVARIANCE_DIAGONAL / largest
        self.samples += 1

    def predict_delta(self, features):
        return float(np.clip(self.weights @ features, -0.50, 0.50))

    def remember(self, temperature, features):
        self.last_temperature = float(temperature)
        self.last_features = features.copy()

    def clear_pending(self):
        self.last_temperature = None
        self.last_features = None


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Select an approximately economical action using online learned dynamics.

    At every step: learn from the preceding issued action, score four physical
    AHU modes and at most three VAV profiles per zone, then choose the lowest
    estimated 20/50-minute discomfort + energy surrogate. Hard guardrails
    prevent idle operation when comfort deviation becomes severe.
    """

    # Four operating points and at most three VAV profiles per running mode:
    # 1 + 3 + 3 + 3 = 10 profiles/zone, versus 49 in Version 6.
    # Two checkpoints versus V6's three: 20 and 50 minutes.
    LEARN_DYNAMICS = True
    OPTIMIZE_ENERGY = True
    PLAN_STEPS = (4, 10)
    DISCOMFORT_WEIGHT = 1.0
    ENERGY_WEIGHT = 1.0

    # "mixed" retains both cooling and local reheat in a single AHU.
    OPERATING_POINTS = (
        OperatingPoint('off', False, 305., 200., 310., False),
        OperatingPoint('cool', True, 289., 760., 310., False),
        OperatingPoint('heat', True, 305., 760., 348., True),
        OperatingPoint('mixed', True, 291., 800., 340., True),
    )

    def __init__(self, building_meta_data, action_spec):
        """Map variable-sized building sensors and actuator locations from metadata."""
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.continuous_count = int(m['n_continuous_action'])
        self.discrete_count = int(m['n_discrete_action'])
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.ahu_supply_actions = [int(x) for x in m['temp_indices']]
        self.ahu_pressure_actions = [int(x) for x in m['pressure_indices']]
        self.ahu_recirc_actions = [int(x) for x in m['recirculation_indices']]
        self.water_actions = [int(x) for x in m['water_indices']]
        self.water_temp_action = self.water_actions[0] if self.water_actions else None
        n_ahu = len(self.ahu_supply_actions)
        self.water_pressure_action = (self.ahu_pressure_actions[n_ahu]
                                      if len(self.ahu_pressure_actions) > n_ahu else None)
        self.water_run_action = self.discrete_count - 1
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        normalizers = m.get('observation_normalization_metadata', {})
        temp_normalizer = normalizers.get('zone_air_temperature_sensor', {})
        self.temp_mean = float(temp_normalizer.get('sample_mean', 0.0))
        self.temp_std = float(temp_normalizer.get('sample_std', 1.0))
        if not np.isfinite(self.temp_std) or self.temp_std <= 0.0:
            self.temp_std = 1.0

        # Outdoor-air observations are NORMALIZED too. Without valid
        # normalization metadata, treat outdoor weather as unknown rather than
        # interpreting a z-score (e.g. -2.0) as a temperature in Kelvin.
        outdoor_normalizer = normalizers.get('outside_air_temperature_sensor', {})
        self.outdoor_mean = float(outdoor_normalizer.get('sample_mean', np.nan))
        self.outdoor_std = float(outdoor_normalizer.get('sample_std', np.nan))
        if (not np.isfinite(self.outdoor_mean) or
                not np.isfinite(self.outdoor_std) or
                self.outdoor_std <= 0.0):
            self.outdoor_mean = np.nan
            self.outdoor_std = np.nan

        def numeric_order(value):
            match = re.search(r'\d+', str(value))
            return (int(match.group()) if match else -1, str(value))

        self.ahus = []
        ahu_ids = sorted(m['ahu_zone_assignments'], key=numeric_order)[:n_ahu]
        for number, ahu_id in enumerate(ahu_ids):
            zones = []
            for room in m['ahu_zone_assignments'][ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                sensor = self.fields.get(vav + '_zone_air_temperature_sensor')
                if sensor is None:
                    continue
                actions = m['zone_action_indices'].get(vav, {})
                zones.append({
                    'name': vav,
                    'sensor': sensor,
                    'heat_setpoint': self.fields.get(
                        vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'cool_setpoint': self.fields.get(
                        vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper_action': actions.get(
                        'supply_air_damper_percentage_command'),
                    'valve_action': actions.get(
                        'reheat_valve_percentage_command'),
                    'model': ZoneResponseModel(),
                })
            self.ahus.append({
                'zones': zones, 'run_action': number,
                'supply_action': self.ahu_supply_actions[number],
                'pressure_action': (
                    self.ahu_pressure_actions[number]
                    if number < len(self.ahu_pressure_actions) else None
                ),
                'recirc_action': (
                    self.ahu_recirc_actions[number]
                    if number < len(self.ahu_recirc_actions) else None
                ),
                'outside_sensor': self.fields.get(
                    'ahs_' + str(ahu_id) + '_outside_air_temperature_sensor'),
            })
        self.reset()

    def reset(self):
        """Isolate episodes, including online model state and pending actions."""
        for ahu in self.ahus:
            for zone in ahu['zones']:
                zone['model'] = ZoneResponseModel()

    @staticmethod
    def _command(fraction):
        """Map physical actuator fraction [0, 1] to normalized [-1, 1]."""
        return float(np.clip(2.0 * fraction - 1.0, -1.0, 1.0))

    @staticmethod
    def _supply_command(kelvin):
        """Map 285..305 Kelvin supply-air setpoint into [-1, 1]."""
        return float(np.clip((kelvin - 285.0) / 10.0 - 1.0, -1.0, 1.0))

    def _fallback_band(self, obs):
        """Use weekday/hour only if current zone setpoint sensors are invalid."""
        def cyclic(sine, cosine, period):
            a, b = self.fields.get(sine), self.fields.get(cosine)
            if a is None or b is None:
                return None
            if not np.isfinite(obs[a]) or not np.isfinite(obs[b]):
                return None
            return (np.arctan2(obs[a], obs[b]) %
                    (2.0 * np.pi)) * period / (2.0 * np.pi)
        hour = cyclic('hod_sin_000', 'hod_cos_000', 24)
        day = cyclic('dow_sin_000', 'dow_cos_000', 7)
        occupied = (hour is not None and day is not None and day < 5 and
                    self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zones(self, obs, ahu, fallback):
        """Denormalize zone temperatures, validate Kelvin setpoints, and learn.

        Observe the temperature response to the PREVIOUS issued action before
        selecting today's candidate. Invalid sensors clear pending transitions.
        """
        result = []
        outdoor_idx = ahu['outside_sensor']
        outdoor = np.nan
        if outdoor_idx is not None and np.isfinite(self.outdoor_std):
            normalized = float(obs[outdoor_idx])
            if np.isfinite(normalized):
                decoded = self.outdoor_mean + normalized * self.outdoor_std
                if 180.0 <= decoded <= 350.0:
                    outdoor = decoded
        for zone in ahu['zones']:
            temp = float(obs[zone['sensor']]) * self.temp_std + self.temp_mean
            if not np.isfinite(temp) or not 240.0 <= temp <= 340.0:
                zone['model'].clear_pending()
                continue
            li, ui = zone['heat_setpoint'], zone['cool_setpoint']
            low = float(obs[li]) if li is not None else np.nan
            high = float(obs[ui]) if ui is not None else np.nan
            if not (np.isfinite(low) and np.isfinite(high) and
                    245.0 <= low < high <= 335.0 and high-low <= 35.0):
                low, high = fallback
            if self.LEARN_DYNAMICS:
                zone['model'].observe(temp)
            result.append({'zone': zone, 'temp': temp, 'low': low,
                           'high': high, 'outdoor': outdoor})
        return result

    @staticmethod
    def _violation(temp, low, high):
        """Comfort deviation in K, with convex penalty beyond the 5K knee."""
        outside = max(low - temp, temp - high, 0.0)
        return outside / 5.0 if outside <= 5.0 else 1.0 + ((outside - 5.0) / 5.0)**2

    @staticmethod
    def _profiles(config, zone):
        """Small, *actuable* VAV grid for one AHU mode.

        With no reheat, search minimum ventilation, medium, and full airflow.
        With reheat, search minimum ventilation, full cooling, and full heat.
        Missing actuator mappings cannot be optimized as if controllable.
        """
        if not config.running:
            return ((0.0, 0.0),)

        can_heat = config.water_on and zone['valve_action'] is not None
        if zone['damper_action'] is None:
            return ((0.1, 0.0), (0.1, 1.0)) if can_heat else ((0.1, 0.0),)
        if can_heat:
            return ((0.1, 0.0), (1.0, 0.0), (1.0, 1.0))
        return ((0.1, 0.0), (0.55, 0.0), (1.0, 0.0))

    def _best_vav(self, state, config):
        """Optimize at most three profiles using a scalar linear model.

        Algebraically equivalent to V6's action-feature model at each fixed
        candidate: delta = base + damper*(cool_gain + valve*heat_gain).
        Compute base and gains once per zone/mode instead of constructing a
        NumPy feature vector for every combination and forecast checkpoint.
        """
        zone, temperature = state['zone'], state['temp']
        model = zone['model']
        weights = model.weights
        outdoor = state['outdoor']
        if not np.isfinite(outdoor):
            outdoor = temperature

        ambient = max(-3.0, min(3.0, (outdoor - temperature) / 20.0))
        base_rate = float(weights[0]) * ambient + float(weights[3])
        cool_gain = 0.0
        heat_gain = 0.0
        if config.running:
            delivered = min(
                config.supply_k, 0.70 * temperature + 0.30 * outdoor)
            cooling_feature = max(
                0.0, min(2.0, (temperature - delivered) / 12.0))
            cool_gain = float(weights[1]) * cooling_feature
            if config.water_on:
                heating_feature = max(
                    0.0, min(2.0, (config.water_k - temperature) / 55.0))
                heat_gain = float(weights[2]) * heating_feature

        best = (float('inf'), 0.0, 0.0)
        for damper, valve in self._profiles(config, zone):
            rate = max(-0.50, min(
                0.50, base_rate + damper * (cool_gain + valve * heat_gain)))
            comfort = sum(self._violation(
                temperature + rate * steps, state['low'], state['high'])
                for steps in self.PLAN_STEPS) / len(self.PLAN_STEPS)
            score = self.DISCOMFORT_WEIGHT * comfort
            if self.OPTIMIZE_ENERGY:
                score += self.ENERGY_WEIGHT * (
                    (0.012 * damper if config.running else 0.0) +
                    (0.022 * damper * valve if config.water_on else 0.0))
            if score < best[0]:
                best = (score, damper, valve)
        return best

    @staticmethod
    def _plant_cost(config):
        """Estimate relative fan, cooling and boiler cost (not true kWh)."""
        if not config.running:
            return 0.0
        fan = 0.035 + 0.035 * (config.fan_pa / 2000.0)**1.5
        cooling = 0.025 * max(0.0, 299.0 - config.supply_k) / 14.0
        heating = (0.025 * max(0.0, config.water_k - 310.0) / 40.0
                   if config.water_on else 0.0)
        return fan + cooling + heating

    @staticmethod
    def _candidate_allowed(config, worst_hot, worst_cold):
        """Keep a corrective mode available for every severe demand.

        Off is forbidden at 2 K violation; one-sided modes are forbidden at
        4 K in the opposite direction. Mixed always remains feasible when
        different zones need simultaneous heating and cooling.
        """
        mode = config.mode
        if mode == 'off' and max(worst_hot, worst_cold) >= 2.0:
            return False
        if worst_hot >= 4.0 and mode == 'heat':
            return False
        if worst_cold >= 4.0 and mode == 'cool':
            return False
        return True

    def _select_operating_point(self, zones):
        """Select an AHU mode plus per-zone VAV profiles by short search.

        Comfort and energy costs are nonnegative, so a partly scored
        candidate can be abandoned as soon as its lower bound exceeds the
        best complete candidate. This never changes the winning result
        within Version 12's reduced action space.
        """
        worst_hot = max((z['temp'] - z['high'] for z in zones), default=0.0)
        worst_cold = max((z['low'] - z['temp'] for z in zones), default=0.0)
        best_cost, best_config, best_vavs = float('inf'), None, None
        zone_count = len(zones)

        for config in self.OPERATING_POINTS:
            if not self._candidate_allowed(config, worst_hot, worst_cold):
                continue
            cost = (self.ENERGY_WEIGHT * self._plant_cost(config)
                    if self.OPTIMIZE_ENERGY else 0.0)
            choices = []
            for state in zones:
                choice = self._best_vav(state, config)
                choices.append(choice)
                cost += choice[0] / zone_count
                if cost >= best_cost:
                    break
            if len(choices) == zone_count and cost < best_cost:
                best_cost, best_config, best_vavs = cost, config, choices
        return best_config, best_vavs

    def _economizer(self, outdoor, zone_temperature, supply, pressure):
        """Choose recirculation when free cooling saves more than fan work.

        Returned +1 requests maximum recirculation, capped by the simulator.
        """
        if not np.isfinite(outdoor) or outdoor >= zone_temperature:
            return 1.0
        if zone_temperature - outdoor <= pressure * 0.00322:
            return 1.0
        fraction_outside = np.clip((zone_temperature - supply) /
                                   max(1e-6, zone_temperature-outdoor), 0.3, 1.0)
        return self._command(1.0 - fraction_outside)

    def _initial_actions(self):
        """Allocate safe idle commands, with correct array sizes and types.

        A normalized continuous zero is the MIDPOINT of a physical range,
        not OFF. +1 requests warmest AHU supply and maximum recirculation.
        """
        continuous = np.full(self.continuous_count, -1.0, dtype=np.float32)
        discrete = np.zeros(self.discrete_count, dtype=np.int32)
        for action_index in self.ahu_supply_actions + self.ahu_recirc_actions:
            continuous[action_index] = 1.0
        return continuous, discrete

    def _apply_ahu_action(self, ahu, states, config, continuous, discrete):
        """Command one AHU's run flag, cooling supply, fan and economizer.

        AHUs cannot heat: thermal reheat occurs downstream in the VAV boxes.
        """
        if config.running:
            discrete[ahu['run_action']] = 1
            continuous[ahu['supply_action']] = self._supply_command(
                config.supply_k)
            if ahu['pressure_action'] is not None:
                continuous[ahu['pressure_action']] = self._command(
                    config.fan_pa / 2000.0)
            if ahu['recirc_action'] is not None and config.supply_k < 300.0:
                outside = states[0]['outdoor']
                mean_zone = float(np.mean([s['temp'] for s in states]))
                continuous[ahu['recirc_action']] = self._economizer(
                    outside, mean_zone, config.supply_k, config.fan_pa)

    def _apply_vav_actions(self, states, config, choices, continuous,
                           heat_demands, pending):
        """Issue achievable zone commands and collect model update inputs.

        We defer saving the model's features until the shared boiler command
        is known: different AHUs can request different water temperatures.
        """
        for state, (_, damper, valve) in zip(states, choices):
            zone = state['zone']
            damper_index = zone['damper_action']
            valve_index = zone['valve_action']
            if valve_index is None:
                valve = 0.0
            if damper_index is None:
                damper = 0.10 if config.running else 0.0

            if damper_index is not None:
                continuous[damper_index] = self._command(damper)
            if valve_index is not None:
                continuous[valve_index] = self._command(valve)
            if config.running and config.water_on and valve > 0.0:
                heat_demands.append((config.water_k, config.fan_pa,
                                     damper * valve))

            pending.append((state, config, damper, valve))

    def _apply_shared_hot_water(self, heat_demands, continuous, discrete):
        """Combine all AHUs' valid reheat requests into one water command.

        Returns the common commanded water temperature, or None if unused.
        """
        if not heat_demands:
            return None

        discrete[self.water_run_action] = 1
        hot_water = max(demand[0] for demand in heat_demands)
        loading = float(np.mean([demand[2] for demand in heat_demands]))
        if self.water_temp_action is not None:
            continuous[self.water_temp_action] = self._command(
                (hot_water - 310.0) / 40.0)
        if self.water_pressure_action is not None:
            continuous[self.water_pressure_action] = self._command(
                max(0.35, loading))
        return hot_water

    @staticmethod
    def _remember_issued_actions(pending, common_water_k):
        """Store the actual *jointly commanded* action for the next RLS update."""
        for state, config, damper, valve in pending:
            heat_enabled = (common_water_k is not None and config.running
                            and config.water_on and valve > 0.0)
            water_k = common_water_k if heat_enabled else config.water_k
            features = state['zone']['model'].features(
                state['temp'], state['outdoor'], float(config.running),
                config.supply_k, damper, valve,
                float(heat_enabled), water_k)
            state['zone']['model'].remember(state['temp'], features)

    def get_action(self, obs):
        """One step: learn previous outcome -> plan -> issue -> remember.

        (1) Decode live sensors and update the online response models.
        (2) Search four AHU modes and at most three VAV profiles per zone.
        (3) Issue HVAC commands and remember model features for the next step.
        (4) Coordinate hot water and return normalized, typed action arrays.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._initial_actions()
        fallback = self._fallback_band(obs)
        heat_demands = []
        pending = []

        for ahu in self.ahus:
            states = self._read_zones(obs, ahu, fallback)
            if not states:
                continue
            config, choices = self._select_operating_point(states)
            self._apply_ahu_action(ahu, states, config, continuous, discrete)
            self._apply_vav_actions(
                states, config, choices, continuous, heat_demands, pending)

        common_water_k = self._apply_shared_hot_water(
            heat_demands, continuous, discrete)
        self._remember_issued_actions(pending, common_water_k)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
